# AIST-FYP Colab Mitigation Evaluation Notebook

This notebook runs **paired baseline vs mitigation** evaluation using existing project scripts.

## Default profile
- Scope: **RAGTruth + CiteEval**
- Variants: **baseline, mitigation_all**
- Runtime target: **smoke test**

## 🔑 Setup Colab Secrets

1. Open the **Secrets** panel (key icon) in Colab sidebar.
2. Add secrets as needed:
   - `DEEPSEEK_API_KEY` (recommended for CiteEval cost)
   - `OPENAI_API_KEY` (optional if using OpenAI provider)
   - `HUGGINGFACE_TOKEN` (optional for gated models)
3. Turn on **Notebook access** for each secret.

The next cell loads these into environment variables automatically.

In [ ]:
# Load Colab secrets into environment variables
import os

try:
    from google.colab import userdata
    secret_keys = ["DEEPSEEK_API_KEY", "OPENAI_API_KEY", "HUGGINGFACE_TOKEN"]
    loaded = []
    for key in secret_keys:
        try:
            value = userdata.get(key)
            if value:
                os.environ[key] = value
                loaded.append(key)
        except Exception:
            pass

    if loaded:
        print("Loaded secrets:", ", ".join(loaded))
    else:
        print("No Colab secrets loaded. Add keys via Secrets panel if needed.")
except ImportError:
    print("Not running in Colab. Export API keys manually in your environment.")

In [ ]:
# ==============================
# Parameters (edit this cell)
# ==============================
REPO_URL = "https://github.com/xiashuidaolaoshuren/AIST-FYP.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/AIST-FYP"
COLAB_ENV_PROJECT = "colab/env"
COLAB_UV_EXTRAS = ["mitigation"]

RUN_RAGTRUTH_MITIGATION = True
RUN_CITEEVAL_MITIGATION = True

# Shared
STRATEGY = "production"            # development | validation | production
VARIANTS = ["baseline", "mitigation_all"]

# RAGTruth mitigation script knobs
RAGTRUTH_SPLIT = "test"            # train | test
RAGTRUTH_EVAL_MODE = "ragtruth_eval"  # ragtruth_eval | normal
RAGTRUTH_MAX_SAMPLES = 10            # smoke test; set None for full split
RAGTRUTH_BATCH_SIZE = 10

# CiteEval mitigation script knobs
CITEEVAL_SYSTEM_SOURCE = "benchmark/CiteEval/data/system_eval/system_eval_examples.json"
CITEEVAL_MAX_SAMPLES = 10            # smoke test
CITEEVAL_PROVIDER = "deepseek"      # deepseek | openai
CITEEVAL_MODEL_NAME = "deepseek-chat"
CITEEVAL_VERSION = "citeeval-auto-12272024"
CITEEVAL_MODULES = "ca,ce,cr_itercoe,cr_editdist"
CITEEVAL_N_THREADS = 8
CITEEVAL_CITED_ONLY = False

# Artifacts reminder (must exist inside repo folder after clone)
# - data/indexes/{STRATEGY}/faiss.index
# - data/indexes/{STRATEGY}/metadata.pkl
# - data/processed/wiki_chunks_{STRATEGY}.jsonl
# - benchmark/RAGTruth/dataset
# - benchmark/CiteEval

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/AIST-FYP-colab-outputs"

In [ ]:
import os
import json
import subprocess
from pathlib import Path

def run(cmd, cwd=None, check=True):
    print(f"\n$ {cmd}")
    process = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    if process.stdout:
        print(process.stdout)
    if process.returncode != 0:
        if process.stderr:
            print(process.stderr)
        if check:
            raise RuntimeError(f"Command failed ({process.returncode}): {cmd}")
    return process

def exists_or_raise(path, msg):
    if not Path(path).exists():
        raise FileNotFoundError(f"{msg}: {path}")

def latest_subdir(parent: Path):
    if not parent.exists():
        return None
    dirs = [p for p in parent.iterdir() if p.is_dir()]
    return max(dirs, key=lambda p: p.name) if dirs else None

In [ ]:
# Mount Drive and clone repo
from google.colab import drive
drive.mount('/content/drive')

if Path(REPO_DIR).exists():
    print(f"Repo dir already exists: {REPO_DIR}")
else:
    run(f"git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")

run("git rev-parse --abbrev-ref HEAD", cwd=REPO_DIR)
run("git log -1 --oneline", cwd=REPO_DIR)

In [ ]:
# Install dependencies
run("python -m pip install -U pip wheel setuptools")
run("python -m pip install -U uv")

uv_project = Path(REPO_DIR) / COLAB_ENV_PROJECT
extras_args = " ".join(f"--extra {extra}" for extra in COLAB_UV_EXTRAS)
sync_cmd = f"uv sync --project {uv_project} {extras_args}"
result = run(sync_cmd, cwd=REPO_DIR, check=False)

if result.returncode == 0:
    uv_python = uv_project / ".venv" / "bin" / "python"
    os.environ["PATH"] = f"{uv_python.parent}:{os.environ.get('PATH', '')}"
    print(f"✅ uv sync complete: {uv_project}")
else:
    print('\n⚠️ uv sync failed. Falling back to pip requirements install...')
    requirements_path = Path(REPO_DIR) / 'requirements.txt'
    pytorch_index = 'https://download.pytorch.org/whl/cu121'
    install_cmd = f"pip install --extra-index-url {pytorch_index} -r {requirements_path}"
    fallback_result = run(install_cmd, cwd=REPO_DIR, check=False)

    if fallback_result.returncode != 0:
        print('\n⚠️ Full requirements install failed. Falling back to Colab-torch-compatible install...')
        filtered = []
        skip_prefixes = ('torch==', 'torchvision==', 'torchaudio==')
        for raw in requirements_path.read_text(encoding='utf-8').splitlines():
            line = raw.strip()
            if not line or line.startswith('#'):
                continue
            if any(line.startswith(prefix) for prefix in skip_prefixes):
                continue
            filtered.append(line)

        temp_req = Path(REPO_DIR) / 'requirements.colab.filtered.txt'
        temp_req.write_text('\n'.join(filtered) + '\n', encoding='utf-8')
        run(f"pip install -r {temp_req}", cwd=REPO_DIR)

# spaCy model required by verifier
run("python -m spacy download en_core_web_sm", cwd=REPO_DIR)

In [ ]:
# Runtime + env setup
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

os.environ["CITEEVAL_PROVIDER"] = CITEEVAL_PROVIDER
os.environ["CITEEVAL_ROOT"] = str(Path(REPO_DIR) / "benchmark/CiteEval")
extra_paths = [str(Path(REPO_DIR) / "benchmark/CiteEval"), str(Path(REPO_DIR) / "benchmark/CiteEval/src")]
existing_pp = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = (existing_pp + os.pathsep if existing_pp else "") + os.pathsep.join(extra_paths)

print("CITEEVAL_PROVIDER =", os.environ.get("CITEEVAL_PROVIDER"))
print("DEEPSEEK_API_KEY set =", bool(os.environ.get("DEEPSEEK_API_KEY")))
print("OPENAI_API_KEY set =", bool(os.environ.get("OPENAI_API_KEY")))

In [ ]:
# Preflight artifact checks
repo = Path(REPO_DIR)
faiss_index = repo / f"data/indexes/{STRATEGY}/faiss.index"
index_meta = repo / f"data/indexes/{STRATEGY}/metadata.pkl"
chunks_file = repo / f"data/processed/wiki_chunks_{STRATEGY}.jsonl"
ragtruth_dataset = repo / "benchmark/RAGTruth/dataset"
citeeval_root = repo / "benchmark/CiteEval"
citeeval_source = repo / CITEEVAL_SYSTEM_SOURCE

exists_or_raise(faiss_index, "Missing FAISS index")
exists_or_raise(index_meta, "Missing index metadata")
exists_or_raise(chunks_file, "Missing processed chunks")
if RUN_RAGTRUTH_MITIGATION:
    exists_or_raise(ragtruth_dataset, "Missing RAGTruth dataset directory")
if RUN_CITEEVAL_MITIGATION:
    exists_or_raise(citeeval_root, "Missing benchmark/CiteEval directory")
    exists_or_raise(citeeval_source, "Missing CiteEval system source JSON")

print("Preflight checks passed.")

In [ ]:
# Run RAGTruth mitigation paired evaluation
if RUN_RAGTRUTH_MITIGATION:
    rag_cmd = [
        "python scripts/evaluate_mitigation_strategy.py",
        "--config config.yaml",
        f"--split {RAGTRUTH_SPLIT}",
        f"--batch-size {RAGTRUTH_BATCH_SIZE}",
        f"--strategy {STRATEGY}",
        f"--ragtruth-eval-mode {RAGTRUTH_EVAL_MODE}",
        "--variants " + " ".join(VARIANTS),
    ]

    if RAGTRUTH_MAX_SAMPLES is not None:
        rag_cmd.append(f"--max-samples {RAGTRUTH_MAX_SAMPLES}")

    run(" ".join(rag_cmd), cwd=REPO_DIR)
else:
    print("Skipped RAGTruth mitigation evaluation.")

In [ ]:
# Run CiteEval mitigation paired evaluation
if RUN_CITEEVAL_MITIGATION:
    cite_cmd = [
        "python scripts/evaluate_mitigation_citebench.py",
        "--config config.yaml",
        f"--strategy {STRATEGY}",
        f"--system-source {CITEEVAL_SYSTEM_SOURCE}",
        "--variants " + " ".join(VARIANTS),
        f"--provider {CITEEVAL_PROVIDER}",
        f"--model-name {CITEEVAL_MODEL_NAME}",
        f"--version {CITEEVAL_VERSION}",
        f"--modules {CITEEVAL_MODULES}",
        f"--n-threads {CITEEVAL_N_THREADS}",
    ]

    if CITEEVAL_MAX_SAMPLES is not None:
        cite_cmd.append(f"--max-samples {CITEEVAL_MAX_SAMPLES}")
    if CITEEVAL_CITED_ONLY:
        cite_cmd.append("--cited-only")

    run(" ".join(cite_cmd), cwd=REPO_DIR)
else:
    print("Skipped CiteEval mitigation evaluation.")

In [ ]:
# Locate latest summaries and copy outputs to Drive
repo = Path(REPO_DIR)
rag_root = repo / "outputs/mitigation_eval"
cite_root = repo / "outputs/mitigation_eval_citebench"

latest_rag = latest_subdir(rag_root)
latest_cite = latest_subdir(cite_root)

if latest_rag:
    print("Latest RAGTruth mitigation run:", latest_rag)
    print("-", latest_rag / "summary.json")
    print("-", latest_rag / "summary.md")
else:
    print("No RAGTruth mitigation run found.")

if latest_cite:
    print("Latest CiteEval mitigation run:", latest_cite)
    print("-", latest_cite / "summary.json")
    print("-", latest_cite / "summary.md")
else:
    print("No CiteEval mitigation run found.")

drive_out = Path(DRIVE_OUTPUT_DIR)
drive_out.mkdir(parents=True, exist_ok=True)

if latest_rag:
    run(f"cp -r {latest_rag} {drive_out}/", cwd=REPO_DIR, check=False)
if latest_cite:
    run(f"cp -r {latest_cite} {drive_out}/", cwd=REPO_DIR, check=False)

print("Drive output directory:", drive_out)

## Scale-up after smoke test

- Set `RAGTRUTH_MAX_SAMPLES = None` for full split.
- Increase `CITEEVAL_MAX_SAMPLES` or set to `None`.
- Expand `VARIANTS` to include ablations when needed (RAGTruth script supports more variants).
- Keep one parameter profile per experiment for reproducibility.